In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)


In [5]:
import os
print("Root:", os.listdir("."))
print("Data folder:", os.listdir("data") if os.path.exists("data") else "data folder does not exist")

Root: ['.config', 'churn-bigml-20.csv', 'churn-bigml-80.csv', 'sample_data', 'drive']
Data folder: data folder does not exist


In [6]:
import os, shutil

os.makedirs("data", exist_ok=True)
shutil.move("churn-bigml-80.csv", "data/churn-bigml-80.csv")
shutil.move("churn-bigml-20.csv", "data/churn-bigml-20.csv")

print(os.listdir("data"))

['churn-bigml-20.csv', 'churn-bigml-80.csv']


In [7]:
train_raw = pd.read_csv("data/churn-bigml-80.csv")
test_raw = pd.read_csv("data/churn-bigml-20.csv")

# Combine into a single raw dataset to demonstrate a full preprocessing pipeline
df = pd.concat([train_raw, test_raw], ignore_index=True)
print("Shape:", df.shape)
df.head()


Shape: (3333, 20)


,State,Account length,Area code,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,Total eve calls,Total eve charge,Total night minutes,Total night calls,Total night charge,Total intl minutes,Total intl calls,Total intl charge,Customer service calls,Churn
0,KS,128,415,No,Yes,25,265.1,110,45.07,197.4,99,16.78,244.7,91,11.01,10.0,3,2.70,1,False
1,OH,107,415,No,Yes,26,161.6,123,27.47,195.5,103,16.62,254.4,103,11.45,13.7,3,3.70,1,False
2,NJ,137,415,No,No,0,243.4,114,41.38,121.2,110,10.30,162.6,104,7.32,12.2,5,3.29,0,False
3,OH,84,408,Yes,No,0,299.4,71,50.90,61.9,88,5.26,196.9,89,8.86,6.6,7,1.78,2,False
4,OK,75,415,Yes,No,0,166.7,113,28.34,148.3,122,12.61,186.9,121,8.41,10.1,3,2.73,3,False


In [8]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3333 entries, 0 to 3332
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   State                   3333 non-null   object 
 1   Account length          3333 non-null   int64  
 2   Area code               3333 non-null   int64  
 3   International plan      3333 non-null   object 
 4   Voice mail plan         3333 non-null   object 
 5   Number vmail messages   3333 non-null   int64  
 6   Total day minutes       3333 non-null   float64
 7   Total day calls         3333 non-null   int64  
 8   Total day charge        3333 non-null   float64
 9   Total eve minutes       3333 non-null   float64
 10  Total eve calls         3333 non-null   int64  
 11  Total eve charge        3333 non-null   float64
 12  Total night minutes     3333 non-null   float64
 13  Total night calls       3333 non-null   int64  
 14  Total night charge      3333 non-null   

In [9]:
print("Missing values per column:")
print(df.isnull().sum())

# Fill any numeric missing values with the column median
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Fill any categorical missing values with the column mode
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()
for c in cat_cols:
    df[c] = df[c].fillna(df[c].mode()[0])

print("\nMissing values after cleaning:", df.isnull().sum().sum())


Missing values per column:
State                     0
Account length            0
Area code                 0
International plan        0
Voice mail plan           0
Number vmail messages     0
Total day minutes         0
Total day calls           0
Total day charge          0
Total eve minutes         0
Total eve calls           0
Total eve charge          0
Total night minutes       0
Total night calls         0
Total night charge        0
Total intl minutes        0
Total intl calls          0
Total intl charge         0
Customer service calls    0
Churn                     0
dtype: int64

Missing values after cleaning: 0


In [10]:
df["International plan"] = df["International plan"].map({"Yes": 1, "No": 0})
df["Voice mail plan"] = df["Voice mail plan"].map({"Yes": 1, "No": 0})
df["Churn"] = df["Churn"].astype(int)

print("State unique values:", df["State"].nunique())
df = pd.get_dummies(df, columns=["State"], drop_first=True)

print("Shape after encoding:", df.shape)
df.head()


State unique values: 51
Shape after encoding: (3333, 69)


,Account length,Area code,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,Total eve calls,Total eve charge,Total night minutes,Total night calls,Total night charge,Total intl minutes,Total intl calls,Total intl charge,Customer service calls,Churn,State_AL,State_AR,State_AZ,State_CA,State_CO,State_CT,State_DC,State_DE,State_FL,State_GA,State_HI,State_IA,State_ID,State_IL,State_IN,State_KS,State_KY,State_LA,State_MA,State_MD,State_ME,State_MI,State_MN,State_MO,State_MS,State_MT,State_NC,State_ND,State_NE,State_NH,State_NJ,State_NM,State_NV,State_NY,State_OH,State_OK,State_OR,State_PA,State_RI,State_SC,State_SD,State_TN,State_TX,State_UT,State_VA,State_VT,State_WA,State_WI,State_WV,State_WY
0,128,415,0,1,25,265.1,110,45.07,197.4,99,16.78,244.7,91,11.01,10.0,3,2.70,1,0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,107,415,0,1,26,161.6,123,27.47,195.5,103,16.62,254.4,103,11.45,13.7,3,3.70,1,0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,137,415,0,0,0,243.4,114,41.38,121.2,110,10.30,162.6,104,7.32,12.2,5,3.29,0,0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
3,84,408,1,0,0,299.4,71,50.90,61.9,88,5.26,196.9,89,8.86,6.6,7,1.78,2,0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
4,75,415,1,0,0,166.7,113,28.34,148.3,122,12.61,186.9,121,8.41,10.1,3,2.73,3,0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False


In [11]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

# Only scale genuinely continuous columns (more than 2 unique values)
scale_cols = [c for c in X.columns if X[c].dtype in [np.float64, np.int64] and X[c].nunique() > 2]

scaler = StandardScaler()
X[scale_cols] = scaler.fit_transform(X[scale_cols])

X[scale_cols].describe().T[["mean", "std"]]


,mean,std
Account length,1.407015e-16,1.00015
Area code,4.199728e-16,1.00015
Number vmail messages,7.674629e-17,1.00015
Total day minutes,-3.165784e-16,1.00015
Total day calls,-1.955964e-16,1.00015
Total day charge,9.699878e-17,1.00015
Total eve minutes,-7.248261e-17,1.00015
Total eve calls,3.346991e-16,1.00015
Total eve charge,1.385697e-16,1.00015
Total night minutes,8.287533e-17,1.00015


In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("\nChurn rate (train):", y_train.mean().round(3))
print("Churn rate (test):", y_test.mean().round(3))


X_train: (2666, 68)
X_test: (667, 68)

Churn rate (train): 0.145
Churn rate (test): 0.145


In [13]:
X_train.assign(Churn=y_train).to_csv("data/processed_train.csv", index=False)
X_test.assign(Churn=y_test).to_csv("data/processed_test.csv", index=False)
print("Saved processed_train.csv and processed_test.csv")


Saved processed_train.csv and processed_test.csv
